# Shopee Review Collector

Thu thập review thật cho 3 shop từ `product_review_mapping.json`. Notebook không yêu cầu hoặc lưu mật khẩu Shopee.

Quy trình:
1. Chạy lần lượt từ trên xuống.
2. Cell **Mở trình duyệt** sẽ mở Chromium với profile riêng.
3. Đăng nhập trực tiếp trong cửa sổ Chromium.
4. Quay lại notebook và chạy cell **Thu thập review**.
5. Chạy cell kiểm tra kết quả rồi đóng trình duyệt.

## 1. Cài đặt lần đầu

Bỏ comment và chạy hai dòng dưới đây nếu môi trường chưa có Playwright. Sau khi cài xong, restart kernel trước khi tiếp tục.

In [ ]:
# %pip install -r requirements.txt
# !python -m playwright install chromium

## 2. Cấu hình và hàm xử lý

In [ ]:
import json
import asyncio
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

from playwright.async_api import async_playwright

# Hoạt động khi Jupyter được mở từ repo root hoặc từ src/crawl_data.
current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / 'Shopee_Dataset').is_dir() and (path / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy project root chứa Shopee_Dataset và src')
DATASET_ROOT = PROJECT_ROOT / 'Shopee_Dataset'
CRAWL_ROOT = PROJECT_ROOT / 'src' / 'crawl_data'

MAPPING_FILE = CRAWL_ROOT / 'product_review_mapping.json'
OUTPUT_DIR = DATASET_ROOT / '3_Unstructured_Data'
PROFILE_DIR = CRAWL_ROOT / '.shopee_browser_profile'
VN_TZ = ZoneInfo('Asia/Ho_Chi_Minh')
REVIEWS_PER_PRODUCT = 5

mapping = json.loads(MAPPING_FILE.read_text(encoding='utf-8'))
SHOP_IDS = sorted({item['Shop_ID'] for item in mapping})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
if len(SHOP_IDS) < 15:
    print(f'WARNING: manifest has {len(SHOP_IDS)} shops; at least 15 are required')
print(f'Products configured: {len(mapping)} across {len(SHOP_IDS)} shops')

In [ ]:
def safe_date(value):
    if not value:
        return None
    try:
        return datetime.fromtimestamp(
            int(value), tz=timezone.utc
        ).astimezone(VN_TZ).date().isoformat()
    except (TypeError, ValueError, OverflowError):
        return None


def normalize_rating(raw, shop_id, product_id, review_id):
    images = raw.get('images') or raw.get('image') or []
    if not isinstance(images, list):
        images = [images] if images else []
    return {
        'Shop_ID': shop_id,
        'product_id': product_id,
        'review_id': review_id,
        'user_name': raw.get('author_username') or raw.get('username'),
        'rating': raw.get('rating_star') or raw.get('rating'),
        'review_time': safe_date(raw.get('ctime') or raw.get('create_time')),
        'review_text': (raw.get('comment') or raw.get('comment_text') or '').strip(),
        'has_image': 1 if images else 0,
        'source_comment_id': raw.get('cmtid') or raw.get('comment_id'),
        'Data_Source': 'Shopee public product review endpoint',
    }


async def fetch_json_in_page(page, url, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            result = await page.evaluate(
        """async (url) => {
            const response = await fetch(url, {
                credentials: 'include',
                headers: {'accept': 'application/json, text/plain, */*'}
            });
            const text = await response.text();
            let body;
            try { body = JSON.parse(text); }
            catch (_) { body = {__raw_text: text}; }
            return {status: response.status, body};
        }""",
                url,
            )
            if result.get('status') not in (429, 500, 502, 503, 504):
                return result
            last_error = RuntimeError(f"retryable HTTP {result.get('status')}")
        except Exception as error:
            last_error = error
        if attempt < attempts:
            await asyncio.sleep(2 ** (attempt - 1))
    raise RuntimeError(f'ratings request failed after {attempts} attempts: {last_error}')


async def fetch_reviews(page, shop_id, item_id, needed):
    collected, seen = [], set()
    offset, page_size = 0, 20
    while len(collected) < needed and offset <= 300:
        base = (
            'https://shopee.vn/api/v2/item/get_ratings'
            f'?flag=1&itemid={item_id}&limit={page_size}'
            f'&offset={offset}&shopid={shop_id}&type=0'
        )
        result = await fetch_json_in_page(page, base + '&filter=1')
        if result.get('status') != 200:
            raise RuntimeError(f"HTTP {result.get('status')} from ratings endpoint")
        body = result.get('body') or {}
        if body.get('error') not in (None, 0):
            raise RuntimeError(f"Shopee API error: {body.get('error')}")
        ratings = ((body.get('data') or {}).get('ratings') or [])
        if not ratings:
            result = await fetch_json_in_page(page, base + '&filter=0')
            body = result.get('body') or {}
            if body.get('error') not in (None, 0):
                raise RuntimeError(f"Shopee API error: {body.get('error')}")
            ratings = ((body.get('data') or {}).get('ratings') or [])
        if not ratings:
            break
        for rating in ratings:
            comment = (rating.get('comment') or '').strip()
            if not comment:
                continue
            key = repr(rating.get('cmtid') or (rating.get('author_username'), rating.get('ctime'), comment))
            if key in seen:
                continue
            seen.add(key)
            collected.append(rating)
            if len(collected) >= needed:
                break
        offset += page_size
    return collected[:needed]

## 3. Mở trình duyệt và đăng nhập

Sau khi cell này chạy xong, đăng nhập Shopee trong cửa sổ Chromium. Không đóng cửa sổ. Khi trang chủ đã sẵn sàng, chạy cell thu thập ở bước 4.

In [ ]:
playwright_runtime = await async_playwright().start()
browser_context = await playwright_runtime.chromium.launch_persistent_context(
    user_data_dir=str(PROFILE_DIR),
    headless=False,
    viewport={'width': 1440, 'height': 1000},
)
page = browser_context.pages[0] if browser_context.pages else await browser_context.new_page()
await page.goto('https://shopee.vn/', wait_until='domcontentloaded', timeout=90_000)
print('Đã mở Shopee. Hãy đăng nhập trực tiếp trong Chromium, sau đó chạy cell tiếp theo.')

## 4. Thu thập và ghi Review JSON

In [ ]:
async def collect_all_reviews(page, mapping, reviews_per_product):
    by_shop = defaultdict(list)
    report = defaultdict(lambda: {
        'products': 0, 'reviews': 0, 'with_image': 0,
        'without_image': 0, 'missing_date': 0,
        'failed_products': [], 'urls': [],
    })
    shop_counters = defaultdict(int)

    for item in mapping:
        shop_id = item['Shop_ID']
        report[shop_id]['products'] += 1
        report[shop_id]['urls'].append(item['product_url'])
        print(f"[{shop_id}] {item['product_id']} - {item['product_name']}")
        try:
            await page.goto(item['product_url'], wait_until='domcontentloaded', timeout=90_000)
            await page.wait_for_timeout(1_800)
            ratings = await fetch_reviews(
                page, int(item['shopee_shop_id']),
                int(item['shopee_item_id']), reviews_per_product,
            )
            if len(ratings) < reviews_per_product:
                print(f"  WARNING: {len(ratings)}/{reviews_per_product} text reviews")
            for raw in ratings:
                shop_counters[shop_id] += 1
                review_id = f"rv_{shop_id}_{shop_counters[shop_id]:06d}"
                row = normalize_rating(raw, shop_id, item['product_id'], review_id)
                if not row['review_text']:
                    continue
                rating = int(row['rating']) if row['rating'] is not None else None
                if rating is None or not 1 <= rating <= 5:
                    continue
                row['rating'] = rating
                by_shop[shop_id].append(row)
                report[shop_id]['reviews'] += 1
                report[shop_id]['with_image'] += row['has_image']
                report[shop_id]['without_image'] += 1 - row['has_image']
                if row['review_time'] is None:
                    report[shop_id]['missing_date'] += 1
            print(f"  collected: {len(ratings)}")
        except Exception as error:
            message = f"{item['product_id']}: {type(error).__name__}: {error}"
            print('  ERROR:', message)
            report[shop_id]['failed_products'].append(message)

    for shop_id in SHOP_IDS:
        destination = OUTPUT_DIR / f'shop_{shop_id}_reviews.json'
        destination.write_text(
            json.dumps(by_shop[shop_id], ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        print(f'Wrote {len(by_shop[shop_id])} reviews -> {destination}')

    lines = ['# Shopee Review Collection Report', '']
    for shop_id in SHOP_IDS:
        result = report[shop_id]
        lines += [
            f'## {shop_id}', '',
            f"- Products attempted: {result['products']}",
            f"- Reviews collected: {result['reviews']}",
            f"- Reviews with image: {result['with_image']}",
            f"- Reviews without image: {result['without_image']}",
            f"- Reviews missing date: {result['missing_date']}",
            f"- Failed products: {len(result['failed_products'])}", '',
            '### Product URLs',
            *[f'- {url}' for url in result['urls']], '',
        ]
        if result['failed_products']:
            lines += ['### Errors', *[f'- {error}' for error in result['failed_products']], '']
    report_path = DATASET_ROOT / 'review_collection_report.md'
    report_path.write_text('\n'.join(lines), encoding='utf-8')
    return by_shop, report


reviews_by_shop, collection_report = await collect_all_reviews(
    page, mapping, REVIEWS_PER_PRODUCT
)

## 5. Kiểm tra schema, ID và liên kết Product

In [ ]:
required_fields = {
    'Shop_ID', 'product_id', 'review_id', 'user_name',
    'rating', 'review_time', 'review_text', 'has_image', 'Data_Source',
}
valid_product_ids = {item['product_id'] for item in mapping}
all_review_ids = []
summary = []

for shop_id in SHOP_IDS:
    path = OUTPUT_DIR / f'shop_{shop_id}_reviews.json'
    rows = json.loads(path.read_text(encoding='utf-8'))
    for row in rows:
        assert required_fields <= set(row), f'Missing fields in {row}'
        assert row['Shop_ID'] == shop_id
        assert row['product_id'] in valid_product_ids
        assert 1 <= int(row['rating']) <= 5
        assert row['has_image'] in (0, 1)
        assert row['review_text'].strip()
        all_review_ids.append(row['review_id'])
    summary.append({
        'Shop_ID': shop_id,
        'Review_Count': len(rows),
        'With_Image': sum(row['has_image'] for row in rows),
    })

assert len(all_review_ids) == len(set(all_review_ids)), 'Duplicate review_id'
summary

## 6. Đóng trình duyệt

In [ ]:
await browser_context.close()
await playwright_runtime.stop()
print('Browser closed.')